<a href="https://colab.research.google.com/github/Ranjith1816/Gen_AI--Assignment-_task/blob/main/Fine_tune_a_transformer_model_to_perform_POS_Tagging_Chunking_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

install dependencies

In [2]:
!pip install -U datasets transformers evaluate seqeval

import libraries

In [1]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate

loading dataset

In [2]:
dataset = load_dataset("wikiann", "en")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [3]:
print(dataset["train"][0])

{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


extract labels

In [4]:
label_list = dataset["train"].features["ner_tags"].feature.names

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


tokenization & label allignment

In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

Tokenization function

In [11]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Preprocessing

In [50]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [51]:
small_train = tokenized_dataset["train"].select(range(5000))
small_val = tokenized_dataset["validation"].select(range(1000))

Model setup

In [52]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training

In [54]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True
)

In [55]:
from transformers import DataCollatorForTokenClassification
data_collator= DataCollatorForTokenClassification(tokenizer)

In [56]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [57]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [58]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.407819,0.287988,0.719443,0.782369,0.749588,0.910802
2,0.266292,0.273936,0.766883,0.813361,0.789439,0.921799


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1250, training_loss=0.3967369255065918, metrics={'train_runtime': 72.0867, 'train_samples_per_second': 138.722, 'train_steps_per_second': 17.34, 'total_flos': 64842462516480.0, 'train_loss': 0.3967369255065918, 'epoch': 2.0})

Evaluation

In [ ]:
trainer.evaluate()

Inference

In [61]:
sentence = "John works at Google in California"

inputs = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)
# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs).logits
predictions = np.argmax(outputs.detach().cpu().numpy(), axis=2)

print([(word, id2label[p]) for word, p in zip(sentence.split(), predictions[0])])

[('John', 'O'), ('works', 'B-PER'), ('at', 'O'), ('Google', 'O'), ('in', 'B-ORG'), ('California', 'O')]


Comparision

🔹 POS Tagging Output (grammer-Level)

| Word       | POS Tag | Meaning                    |
| ---------- | ------- | -------------------------- |
| John       | NNP     | Proper Noun (Person name)  |
| works      | VBZ     | Verb (3rd person singular) |
| at         | IN      | Preposition                |
| Google     | NNP     | Proper Noun (Organization) |
| in         | IN      | Preposition                |
| California | NNP     | Proper Noun (Place)        |


🔹 Chunking / NER Output (Phrase-Level)

| Word       | Chunk Tag | Meaning                    |
| ---------- | --------- | -------------------------- |
| John       | B-PER     | Beginning of Person entity |
| works      | O         | Outside any entity         |
| at         | O         | Outside any entity         |
| Google     | B-ORG     | Beginning of Organization  |
| in         | O         | Outside any entity         |
| California | B-LOC     | Beginning of Location      |


| Feature           | POS Tagging             | Chunking                 |
| ----------------- | ----------------------- | ------------------------ |
| Level of Analysis | Word-level              | Phrase-level             |
| Purpose           | Grammar understanding   | Structure understanding  |
| Output Type       | POS labels (NN, VB, JJ) | BIO tags (B-NP, I-NP, O) |
| Complexity        | Easy                    | Medium                   |
| Context Use       | Limited                 | Higher                   |
| Example           | “runs → Verb”           | “New York → Location”    |
| Use Case          | Syntax analysis         | Information extraction   |




Key differences
| Feature           | POS Tagging             | Chunking                 |
| ----------------- | ----------------------- | ------------------------ |
| Level of Analysis | Word-level              | Phrase-level             |
| Purpose           | Grammar understanding   | Structure understanding  |
| Output Type       | POS labels (NN, VB, JJ) | BIO tags (B-NP, I-NP, O) |
| Complexity        | Easy                    | Medium                   |
| Context Use       | Limited                 | Higher                   |
| Example           | “runs → Verb”           | “New York → Location”    |
| Use Case          | Syntax analysis         | Information extraction   |


Report

##Difference Between POS Tagging and Chunking
POS Tagging:

Part-of-Speech tagging assigns grammatical categories to each word, such as noun, verb, or adjective.
It focuses on understanding the syntactic role of individual words.

Example:

“runs” → Verb

“John” → Proper Noun

Chunking (Phrase Detection):

Chunking groups words into meaningful phrases or entities, such as person names, locations, or organizations.
It focuses on contextual grouping rather than individual word roles.

Example:

“John” → Person

“Google” → Organization


Challenges faced

* Initially, standard datasets like CoNLL-2003 and Universal Dependencies failed to load due to compatibility issues in the environment. This required switching to a more stable dataset (WikiANN).

* Transformer models break words into subwords, which creates a mismatch between tokens and labels.
Handling this required:

Mapping labels correctly
Ignoring subword tokens using -100

* training transformer models in Google Colab required:

Reducing dataset size
Optimizing batch size
Limiting epochs



Key Observations & Insights:

* BERT captures the meaning of words based on surrounding context, unlike traditional NLP models that process text in a fixed direction.

* Even with a small number of training epochs, BERT delivers strong results, making it efficient for practical use.

* Using pretrained models reduces the need for training from scratch and significantly improves accuracy with limited data.

* Token classification plays a crucial role in systems like:
Chatbots
Search engines
Information extraction tools